In [30]:
# 데이터 처리
import pandas as pd

# 데이터를 학습용과 평가용으로 분리
from sklearn.model_selection import train_test_split

# 도서 제목을 TF-IDF 숫자 벡터로 변환
from sklearn.feature_extraction.text import TfidfVectorizer

# 텍스트 분류 모델
from sklearn.naive_bayes import MultinomialNB

# 모델 성능 평가
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

In [31]:
# 분석에 사용할 CSV 파일 경로
DATA_PATH = "book_bestseller_clean.csv"

# CSV 파일 불러오기
df_books = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
)

# 데이터 크기와 컬럼 확인
print("데이터 크기:", df_books.shape)
print("컬럼:", df_books.columns.tolist())

# 모델링에 사용할 상품명과 분야 확인
df_books[["상품명", "분야"]].head(10)

데이터 크기: (989, 8)
컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']


,상품명,분야
0,"세네카, 오늘을 빼앗기고 있는 당신에게",인문
1,홍정기의 장수근육 혁명,건강
2,싯다르타,소설
3,머니 트렌드 2027,경제/경영
4,흔한남매 23,어린이(초등)
5,2026 해커스 투자자산운용사 실전동형모의고사 10회분+리얼 기출족보 3종,취업/수험서
6,시대예보: 수고인류의 시간,경제/경영
7,리더는 언제 차이를 만들어내는가,인문
8,판매의 법칙,경제/경영
9,니체의 초월자,인문


### 실습 3. 모델링 데이터 정리하기

In [32]:
# 모델링에 필요한 상품명과 분야만 복사
df_model = df_books[["상품명", "분야"]].copy()

# 결측값, 자료형, 앞뒤 공백 정리
for col in ["상품명", "분야"]:
    df_model[col] = (
        df_model[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

# 상품명이나 분야가 비어 있는 행 제거
df_model = df_model[
    (df_model["상품명"] != "") &
    (df_model["분야"] != "")
].reset_index(drop=True)

# 정리된 데이터 크기 확인
print("모델링 데이터 크기:", df_model.shape)

모델링 데이터 크기: (927, 2)


In [33]:
df_model.head()

,상품명,분야
0,"세네카, 오늘을 빼앗기고 있는 당신에게",인문
1,홍정기의 장수근육 혁명,건강
2,싯다르타,소설
3,머니 트렌드 2027,경제/경영
4,흔한남매 23,어린이(초등)


In [34]:
# 실습 4. 분야 분포 확인하기

# 전체 분야 종류 수 확인
print("분야 종류 수:", df_model["분야"].nunique())

# 분야별 도서 수가 많은 순서대로 확인
df_model["분야"].value_counts().head(20)

분야 종류 수: 36


분야
소설          152
취업/수험서      116
인문          107
시/에세이        73
경제/경영        68
외국어          67
자기계발         63
어린이(초등)      57
만화           34
컴퓨터/IT       22
유아(0~7세)     15
정치/사회        15
청소년          13
가정/육아        13
역사/문화        13
잡지           13
과학           11
종교           10
건강            9
책향            9
Name: count, dtype: int64

In [35]:
# 분야별 데이터 개수 계산
class_counts = df_model["분야"].value_counts()

# 데이터가 2개 미만인 분야 확인
class_counts[class_counts < 2]

분야
독서용품        1
예술/건축       1
실용서/예술      1
ELT/수험서     1
디지털-tech    1
인문/사회       1
필기구         1
Name: count, dtype: int64

In [36]:
class_counts

분야
소설           152
취업/수험서       116
인문           107
시/에세이         73
경제/경영         68
외국어           67
자기계발          63
어린이(초등)       57
만화            34
컴퓨터/IT        22
유아(0~7세)      15
정치/사회         15
청소년           13
가정/육아         13
역사/문화         13
잡지            13
과학            11
종교            10
건강             9
책향             9
취미/실용/스포츠      7
요리             7
교보문고 굿즈        5
여행             5
예술/대중문화        4
유아/아동/청소년      4
기술/공학          3
문학             3
가요             2
독서용품           1
예술/건축          1
실용서/예술         1
ELT/수험서        1
디지털-tech       1
인문/사회          1
필기구            1
Name: count, dtype: int64

In [37]:
# Train/Test 분리가 가능한 분야만 선택
valid_classes = class_counts[
    class_counts >= 2
].index

# 데이터가 2개 이상인 분야만 남기기
df_model = df_model[
    df_model["분야"].isin(valid_classes)
].reset_index(drop=True)

# 정리 결과 확인
print("정리 후 데이터 크기:", df_model.shape)
print("정리 후 분야 종류 수:", df_model["분야"].nunique())

정리 후 데이터 크기: (920, 2)
정리 후 분야 종류 수: 29


### 실습 5. X와 y 정의하기

In [38]:
# 모델의 입력 데이터: 도서 제목
X = df_model["상품명"]

# 모델이 예측할 정답: 도서 분야
y = df_model["분야"]

# X와 y의 데이터 개수 확인
print("X 개수:", len(X))
print("y 개수:", len(y))

# 첫 번째 입력과 정답 확인
print("첫 번째 X:", X.iloc[0])
print("첫 번째 y:", y.iloc[0])

X 개수: 920
y 개수: 920
첫 번째 X: 세네카, 오늘을 빼앗기고 있는 당신에게
첫 번째 y: 인문


### 실습 6. train/test 분리하기

In [39]:
# 전체 데이터를 학습용 80%, 평가용 20%로 분리
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 분리된 데이터 크기 확인
print("Train X:", X_train.shape)
print("Test X :", X_test.shape)
print("Train y:", y_train.shape)
print("Test y :", y_test.shape)

Train X: (736,)
Test X : (184,)
Train y: (736,)
Test y : (184,)


### 실습 7~9. Train에만 TF-IDF 학습하기

In [40]:
# TF-IDF 변환기 생성
tfidf = TfidfVectorizer()

# Train 데이터로 단어 사전과 TF-IDF 규칙 학습
X_train_tfidf = tfidf.fit_transform(X_train)

# Train에서 학습한 규칙으로 Test 데이터 변환
X_test_tfidf = tfidf.transform(X_test)

# 변환된 TF-IDF 행렬 크기 확인
print("Train TF-IDF:", X_train_tfidf.shape)
print("Test TF-IDF :", X_test_tfidf.shape)

Train TF-IDF: (736, 1783)
Test TF-IDF : (184, 1783)


### 실습 10~11. Naive Bayes 학습 및 예측

In [41]:
# Multinomial Naive Bayes 모델 생성
model = MultinomialNB()

# Train 데이터와 실제 분야를 이용해 모델 학습
model.fit(X_train_tfidf, y_train)

# Test 데이터의 분야 예측
y_pred = model.predict(X_test_tfidf)

# 예측 결과 확인
print("예측 개수:", len(y_pred))
print("예측 결과 일부:", y_pred[:10])

예측 개수: 184
예측 결과 일부: ['소설' '인문' '외국어' '소설' '소설' '외국어' '취업/수험서' '소설' '컴퓨터/IT' '인문']


### 실습 11. 실제 분야와 예측 분야 비교하기

In [42]:
# 상품명, 실제 분야, 예측 분야를 하나의 표로 생성
result = pd.DataFrame({
    "상품명": X_test.reset_index(drop=True),
    "실제_분야": y_test.reset_index(drop=True),
    "예측_분야": y_pred,
})

# 앞의 20개 예측 결과 확인
result.head(20)

,상품명,실제_분야,예측_분야
0,드라큘라,소설,소설
1,예측의 기술,자기계발,인문
2,JLPT(일본어능력시험) 한권으로 끝내기 N1,외국어,외국어
3,잘잘잘 건강,건강,소설
4,죽음의 수용소에서,시/에세이,소설
5,ETS 토익 정기시험 기출문제집 1000 Vol 5 RC,외국어,외국어
6,2027 단계별기출 핵심지문 박철한 경찰헌법 OX 시즌5,취업/수험서,취업/수험서
7,The Scent of Page : 차량용 방향제(개선판),책향,소설
8,"요즘 AI 루프 엔지니어링, 클로드 코드, 스킬, MCP, 훅, 컨텍스트, 하네스,...",컴퓨터/IT,컴퓨터/IT
9,명상록,인문,인문


### 실습 12. Accuracy 확인하기

In [43]:
# 전체 Test 데이터에 대한 정확도 계산
accuracy = accuracy_score(
    y_test,
    y_pred,
)

# 정확도와 정답 개수 출력
print(f"Accuracy: {accuracy:.4f}")
print(
    "맞힌 개수:",
    (y_test.reset_index(drop=True) == y_pred).sum(),
)
print("전체 개수:", len(y_test))

Accuracy: 0.3859
맞힌 개수: 71
전체 개수: 184


### 실습 13. Classification Report

In [44]:
# 분야별 precision, recall, F1-score 계산
report = classification_report(
    y_test,
    y_pred,
    zero_division=0,
)

# 분야별 평가 결과 출력
print(report)

              precision    recall  f1-score   support

       가정/육아       0.00      0.00      0.00         3
          건강       0.00      0.00      0.00         2
       경제/경영       1.00      0.15      0.27        13
          과학       0.00      0.00      0.00         2
     교보문고 굿즈       0.00      0.00      0.00         1
       기술/공학       0.00      0.00      0.00         1
          만화       0.00      0.00      0.00         7
          문학       0.00      0.00      0.00         1
          소설       0.22      0.93      0.36        30
       시/에세이       0.00      0.00      0.00        15
     어린이(초등)       0.00      0.00      0.00        11
          여행       0.00      0.00      0.00         1
       역사/문화       0.00      0.00      0.00         3
     예술/대중문화       0.00      0.00      0.00         1
         외국어       1.00      0.69      0.82        13
          요리       0.00      0.00      0.00         1
    유아(0~7세)       0.00      0.00      0.00         3
   유아/아동/청소년       0.00    

### 실습 14. Confusion Matrix 확인하기

In [45]:
# 모델이 학습한 분야 목록 가져오기
labels = model.classes_

# 실제 분야와 예측 분야의 혼동 행렬 생성
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels,
)

# 보기 쉽도록 DataFrame으로 변환
df_cm = pd.DataFrame(
    cm,
    index=labels,
    columns=labels,
)

# Confusion Matrix 크기 확인
print("Confusion Matrix 크기:", df_cm.shape)

# Confusion Matrix 출력
df_cm

Confusion Matrix 크기: (29, 29)


,가요,가정/육아,건강,경제/경영,과학,교보문고 굿즈,기술/공학,만화,문학,소설,...,인문,자기계발,잡지,정치/사회,종교,책향,청소년,취미/실용/스포츠,취업/수험서,컴퓨터/IT
가요,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
가정/육아,0,0,0,0,0,0,0,0,0,3,...,0,0,0,0,0,0,0,0,0,0
건강,0,0,0,0,0,0,0,0,0,2,...,0,0,0,0,0,0,0,0,0,0
경제/경영,0,0,0,2,0,0,0,0,0,9,...,1,0,0,0,0,0,0,0,1,0
과학,0,0,0,0,0,0,0,0,0,2,...,0,0,0,0,0,0,0,0,0,0
교보문고 굿즈,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
기술/공학,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
만화,0,0,0,0,0,0,0,0,0,7,...,0,0,0,0,0,0,0,0,0,0
문학,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
소설,0,0,0,0,0,0,0,0,0,28,...,2,0,0,0,0,0,0,0,0,0


### 실습 15. 오분류 사례 확인하기

In [46]:
# 실제 분야와 예측 분야가 다른 데이터만 선택
misclassified = result[
    result["실제_분야"] != result["예측_분야"]
].copy()

# 오분류 개수 확인
print("오분류 수:", len(misclassified))

# 대표 오분류 사례 확인
misclassified.head(20)

오분류 수: 113


,상품명,실제_분야,예측_분야
1,예측의 기술,자기계발,인문
3,잘잘잘 건강,건강,소설
4,죽음의 수용소에서,시/에세이,소설
7,The Scent of Page : 차량용 방향제(개선판),책향,소설
11,사라진 근대사 100장면 1: 몰락의 시대,역사/문화,소설
14,말의 권력,정치/사회,소설
16,코스모스(100만부 기념판),과학,소설
18,호랑이를 부탁해,어린이(초등),소설
19,나는 도대체 왜 눈치를 볼까,자기계발,소설
20,원씽(The One Thing)(60만 부 기념 스페셜 에디션),자기계발,소설


### 오분류 사례 분석

1. **상품명:** 잘잘잘 건강  
   - **실제 분야:** 건강  
   - **예측 분야:** 소설  
   - **틀린 이유:** 건강 분야의 학습 데이터가 적고, 제목이 짧아 분야를 판단할 수 있는 단어가 충분하지 않았기 때문에 소설로 잘못 예측한 것 같다.

2. **상품명:** 호랑이를 부탁해  
   - **실제 분야:** 어린이(초등)  
   - **예측 분야:** 소설  
   - **틀린 이유:** 제목에 어린이 도서임을 나타내는 단어가 없고 이야기 형식의 소설처럼 보일 수 있어서 소설로 잘못 예측한 것 같다.

3. **상품명:** 주술회전 모듈로 2: 두라 바르 보비디 메치카  
   - **실제 분야:** 만화  
   - **예측 분야:** 소설  
   - **틀린 이유:** 제목에 만화임을 직접 나타내는 단어가 없고, 만화 분야의 학습 데이터가 소설보다 적어서 소설로 잘못 예측한 것 같다.

### 실습 16. 예측 결과 저장하기

In [47]:
# 전체 예측 결과를 CSV 파일로 저장
result.to_csv(
    "chapter04_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

# 오분류 결과만 별도의 CSV 파일로 저장
misclassified.to_csv(
    "chapter04_misclassified.csv",
    index=False,
    encoding="utf-8-sig",
)

# 저장 완료 확인
print("예측 결과 저장 완료")
print("오분류 결과 저장 완료")

예측 결과 저장 완료
오분류 결과 저장 완료


In [48]:
# 저장한 전체 예측 결과 다시 불러오기
saved_predictions = pd.read_csv(
    "chapter04_predictions.csv",
    encoding="utf-8-sig",
)

# 저장한 오분류 결과 다시 불러오기
saved_misclassified = pd.read_csv(
    "chapter04_misclassified.csv",
    encoding="utf-8-sig",
)

# 행과 열의 개수 및 컬럼 확인
print("전체 예측 파일:", saved_predictions.shape)
print("오분류 파일:", saved_misclassified.shape)
print("컬럼:", saved_predictions.columns.tolist())

# 저장된 데이터 일부 확인
saved_predictions.head()

전체 예측 파일: (184, 3)
오분류 파일: (113, 3)
컬럼: ['상품명', '실제_분야', '예측_분야']


,상품명,실제_분야,예측_분야
0,드라큘라,소설,소설
1,예측의 기술,자기계발,인문
2,JLPT(일본어능력시험) 한권으로 끝내기 N1,외국어,외국어
3,잘잘잘 건강,건강,소설
4,죽음의 수용소에서,시/에세이,소설


### 실습 17. 새로운 도서 제목 예측하기

In [49]:
# 모델이 예측할 새로운 도서 제목
new_titles = [
    "파이썬으로 시작하는 데이터 분석",
    "처음 배우는 주식 투자",
    "마음을 이해하는 심리학",
]

# 기존 TF-IDF 규칙으로 새로운 제목 변환
new_vectors = tfidf.transform(new_titles)

# 학습된 모델로 새로운 제목의 분야 예측
new_predictions = model.predict(new_vectors)

# 새로운 제목과 예상 분야를 표로 생성
new_result = pd.DataFrame({
    "상품명": new_titles,
    "예상_분야": new_predictions,
})

# 새로운 제목의 예측 결과 확인
new_result

,상품명,예상_분야
0,파이썬으로 시작하는 데이터 분석,소설
1,처음 배우는 주식 투자,소설
2,마음을 이해하는 심리학,인문


In [50]:
tfidf.transform(new_titles)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6 stored elements and shape (3, 1783)>

### 실습 18. 모델의 한계 정리

### 모델의 한계

- 현재 모델은 도서 제목만 사용하므로 책 소개, 저자, 출판사, 키워드 등의 정보는 반영하지 못한다.
- 제목만으로 분야가 명확하지 않은 도서는 정확하게 분류하기 어렵다.
- 분야별 데이터 수가 달라 데이터가 많은 소설 분야로 예측이 치우치는 경향이 나타났다.
- 현재 베스트셀러 데이터와 분야 라벨을 기준으로 학습했으므로 모든 도서에 동일한 성능을 보장할 수 없다.
- 모델의 예측은 공식 분야 분류가 아니라 학습 데이터의 단어 패턴을 바탕으로 한 결과이다.

### 실습 19. 데이터 누수 최종 점검

### 데이터 누수 최종 점검

- [x] TF-IDF를 적용하기 전에 Train/Test 데이터를 먼저 분리했다.
- [x] `fit_transform()`은 `X_train`에만 적용했다.
- [x] `X_test`에는 학습된 TF-IDF의 `transform()`만 적용했다.
- [x] 모델은 Train 데이터만 사용해 학습했다.
- [x] 새로운 도서 제목에도 기존 TF-IDF의 `transform()`을 사용했다.

Test 데이터의 정보를 학습 과정에 미리 사용하지 않았으므로 데이터 누수를 방지했다.

### 실습 20. 자주 만나는 오류 정리

### 실습 중 확인한 오류와 주의점

- `stratify=y`를 사용하기 전에 분야별 데이터 수를 확인했다.
- 데이터가 1개뿐인 7개 분야는 Train/Test로 나눌 수 없어 모델링 데이터에서 제외했다.
- `X`와 `y`는 같은 `df_model`에서 만들어 길이를 동일하게 유지했다.
- Test 데이터와 새로운 제목에는 `fit_transform()`이 아니라 `transform()`을 사용했다.
- 새로운 제목의 희소 행렬 출력은 오류가 아니라 TF-IDF 변환 결과이다.

### 실습 22. Chapter 04 결과 정리

## Chapter 04 결과

### 데이터

- 원본 데이터: 989권
- 모델링 데이터: 920권
- 입력 X: 상품명
- 정답 y: 분야
- 분야 수: 29개
- Train/Test: 736개 / 184개

### 모델

- 텍스트 변환: `TfidfVectorizer`
- 분류 모델: `MultinomialNB`
- Train 데이터에만 TF-IDF를 fit하고, Test 데이터에는 transform만 적용했다.

### 평가

- Accuracy: 0.3859
- 전체 Test 데이터: 184권
- 정답: 71권
- 오분류: 113권
- 외국어와 취업/수험서는 비교적 잘 분류했다.
- 소설의 recall은 높았지만 다른 분야까지 소설로 예측하는 경향이 나타났다.

### 오분류 사례

1. **잘잘잘 건강**
   - 실제 분야: 건강
   - 예측 분야: 소설
   - 건강 분야의 학습 데이터가 적고, 제목이 짧아 분야를 판단할 단어가 충분하지 않았기 때문에 잘못 예측한 것 같다.

2. **코스모스(100만부 기념판)**
   - 실제 분야: 과학
   - 예측 분야: 소설
   - 제목만으로 과학 분야라는 사실이 명확하게 드러나지 않아 잘못 예측한 것 같다.

3. **주술회전 모듈로 2: 두라 바르 보비디 메치카**
   - 실제 분야: 만화
   - 예측 분야: 소설
   - 제목에 만화임을 직접 나타내는 단어가 없고, 만화의 학습 데이터가 소설보다 적어서 잘못 예측한 것 같다.

### 한계

- 도서 제목만 사용해 책의 내용과 세부 정보를 반영하지 못했다.
- 분야별 데이터 수가 달라 소설처럼 데이터가 많은 분야로 예측이 치우쳤다.
- 현재 베스트셀러 데이터와 분야 라벨 안에서 평가한 결과이므로 모든 도서를 대표하지 않는다.

### 한 문장 정리

원본 데이터를 Train/Test로 먼저 나누고, Train 데이터에만 TF-IDF를 학습한 뒤 Naive Bayes로 도서 분야를 예측하고 오분류 사례를 확인했다.